In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump

DATA_PATH = Path('/home/admin/main/ucsd-phys-139-final/data/features.csv')
MODELS_DIR = Path('/home/admin/main/ucsd-phys-139-final/models')
ARTIFACTS_DIR = Path('/home/admin/main/ucsd-phys-139-final/data')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CONTAMINATION = 0.05
np.random.seed(RANDOM_STATE)

In [ ]:
assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
id_col = 'star_id' if 'star_id' in df.columns else None
feature_cols = [c for c in df.columns if c != id_col] if id_col else list(df.columns)

X = df[feature_cols].copy()

ids = df[id_col].copy() if id_col else pd.Series(np.arange(len(df)), name='row_id')

X = X.apply(pd.to_numeric, errors='coerce')
X = X.apply(lambda s: s.fillna(s.median()))

X.describe().T


In [ ]:
iso = IsolationForest(
    n_estimators=300,
    max_samples='auto',
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X)

iso


In [ ]:
anomaly_score = -iso.score_samples(X)
preds = iso.predict(X)
is_anomaly = (preds == -1)

results = pd.DataFrame({
    'id': ids.values,
    'anomaly_score': anomaly_score,
    'is_anomaly': is_anomaly,
})

TOP_N = 20
results_sorted = results.sort_values('anomaly_score', ascending=False)
results_sorted.head(TOP_N)


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(anomaly_score, bins=40, kde=True)
plt.xlabel('Anomaly score (higher = more anomalous)')
plt.ylabel('Count')

q = np.quantile(anomaly_score, 1 - CONTAMINATION)
plt.axvline(q, color='red', linestyle='--', label=f'Quantile @ 1-CONTAM ({1-CONTAMINATION:.2f})')
plt.legend()
plt.tight_layout()
plt.show()

q


In [ ]:
model_path = MODELS_DIR / 'isolation_forest.joblib'
scores_path = ARTIFACTS_DIR / 'anomaly_scores.csv'

dump(iso, model_path)
results_sorted.to_csv(scores_path, index=False)

model_path, scores_path
